# YOLOv8s Training — Guns & Knives Weapon Detection

Fine-tunes **YOLOv8s** (11M params) on the 2-class weapon dataset (pistol, knife).  
Expected improvement: mAP50 **0.724 → 0.78–0.85**

**Colab**: Runtime → Change runtime type → T4 GPU  
**Local**: Skip the Drive mount cell.

In [ ]:
!pip install ultralytics --quiet

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Local mode.')

In [ ]:
import sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # ── EDIT THIS: path to your project folder on Google Drive ──
    PROJECT_ROOT = Path('/content/drive/MyDrive/majorProject')
else:
    PROJECT_ROOT = Path('..').resolve()

DATA_YAML = PROJECT_ROOT / 'datasets/guns-knives/combined_gunsnknifes/data.yaml'
MODEL_OUT = PROJECT_ROOT / 'models/knifes&pistol'
RESULTS   = PROJECT_ROOT / 'results'

assert DATA_YAML.exists(), f'data.yaml not found: {DATA_YAML}'
MODEL_OUT.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_YAML   :', DATA_YAML)

In [ ]:
import torch
GPU     = torch.cuda.is_available()
DEVICE  = 0 if GPU else 'cpu'
BATCH   = 'auto' if GPU else 4
WORKERS = 8 if GPU else 2
print(f'GPU: {GPU}  |  batch={BATCH}  |  workers={WORKERS}')
if GPU:
    print(f'GPU name : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from ultralytics import YOLO

# yolov8s.pt auto-downloaded on first run (~22 MB)
model = YOLO('yolov8s.pt')

model.train(
    data          = str(DATA_YAML),
    epochs        = 100,
    patience      = 20,
    imgsz         = 640,
    batch         = BATCH,
    device        = DEVICE,
    workers       = WORKERS,
    project       = str(RESULTS),
    name          = 'yolo_train_results',
    exist_ok      = True,
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3.0,
    mosaic        = 1.0,
    mixup         = 0.15,
    flipud        = 0.1,
    fliplr        = 0.5,
    degrees       = 10.0,
    translate     = 0.1,
    scale         = 0.5,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    copy_paste    = 0.1,
    plots         = True,
    save          = True,
    save_period   = 10,
    val           = True,
    verbose       = True,
)
print('Training complete.')

In [ ]:
dest = MODEL_OUT / 'best_v2.pt'
for src in sorted(RESULTS.glob('yolo_train_results*/weights/best.pt')):
    shutil.copy2(src, dest)
    print(f'Saved → {dest}')
    break
else:
    print('best.pt not found — check results/yolo_train_results/weights/')

In [ ]:
from ultralytics import YOLO
best = YOLO(str(MODEL_OUT / 'best_v2.pt'))
r = best.val(data=str(DATA_YAML), split='test', imgsz=640, device=DEVICE, conf=0.25, iou=0.45)
print(f'mAP50    : {r.box.map50:.4f}')
print(f'Precision: {r.box.mp:.4f}')
print(f'Recall   : {r.box.mr:.4f}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_files = list(RESULTS.glob('yolo_train_results*/results.csv'))
if csv_files:
    df = pd.read_csv(csv_files[0])
    df.columns = df.columns.str.strip()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    if 'train/box_loss' in df.columns:
        ax1.plot(df['epoch'], df['train/box_loss'], label='Train')
        ax1.plot(df['epoch'], df['val/box_loss'],   label='Val')
        ax1.set_title('Box Loss'); ax1.legend()
    col = 'metrics/mAP50(B)'
    if col in df.columns:
        ax2.plot(df['epoch'], df[col], color='green', label='mAP50')
        ax2.axhline(0.724, color='red', linestyle='--', label='Baseline (yolov8n)')
        ax2.set_title('mAP50'); ax2.legend()
    plt.tight_layout()
    out = RESULTS / 'yolo_training_curves.png'
    plt.savefig(str(out), dpi=120)
    plt.show()
    print(f'Saved → {out}')